<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_34_django_forms/note_lesson_34_forms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Урок 34 — Django: форми, CRUD і PRG

| Крок | Що робимо |
|---|---|
| 0 | запускаємо `django_bootstrap_project` (рефакторинг 1 уроку) у ноутбуці |
| 1 | `ModelForm`: валідація і `cleaned_data` (вправи 1–2) |
| 2 | view передає дані в шаблон (вправа 3) |
| 3 | створення: помилки → та сама форма, успіх → redirect (вправи 4–5) |
| 4 | редагування і видалення (вправи 6–7) |
| 5 | CSRF (вправа 8) |
| 6 | знайди помилку: без PRG (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія, шаблони й скріншоти — у книзі курсу: [Урок 34](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/); поглиблено — кроки 2 і 4 [Django-книги](https://nikoriakviktot.github.io/notes_chat_app/tutorials/).

---
## 0. Проєкт у ноутбуці

Проєкт — `django_bootstrap_project` з уроку: `NoteForm` (`hello_app/forms.py`), CRUD-views за шаблоном PRG і шаблони на Bootstrap. Це код старого курсу, який на сторінці уроку розібрано як рефакторинг проєкту уроку 33. У Colab клітинка нижче завантажить лише папку уроку і встановить залежності з `requirements.txt` проєкту.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_34_django_forms"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("django_bootstrap_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"django": "Django>=5.2,<6", "django_bootstrap5": "django-bootstrap5",
            "unfold": "django-unfold", "debug_toolbar": "django-debug-toolbar"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

os.chdir("django_bootstrap_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 33)
django.setup()

from django.test import Client, RequestFactory
from django.test.utils import setup_test_environment

from hello_app.forms import NoteForm
from hello_app.models import Note

setup_test_environment()                               # хост testserver і контекст шаблонів у відповідях
Note.objects.all().delete()
Note.objects.create(title="Купити молоко", content="2 л")
Note.objects.create(title="Вивчити Django Forms", content="ModelForm, is_valid, PRG")
Note.objects.create(title="Пароль від Wi-Fi", content="coffee2026")
client = Client()
print("нотаток:", Note.objects.count())

---
## 1. ModelForm

`NoteForm(data=...)` → `is_valid()` → `cleaned_data` або `errors`. Дані з браузера — завжди **рядки**.

### Вправа 1

Перевір форму з даними `{"title": "  Здати звіт  ", "content": ""}`: результат `is_valid()` — у `ok`, очищені дані — у `data`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
form = NoteForm(data={"title": "  Здати звіт  ", "content": ""})
ok = form.is_valid()
data = form.cleaned_data
# END SOLUTION

assert ok is True
assert data["title"] == "Здати звіт", "CharField прибирає пробіли на краях"
assert data["content"] == "", "content необов'язковий: blank=True у моделі"
print(data)
print("✅ Вправа 1 пройдена")

### Вправа 2

Зібери в множину `bad_fields` назви полів з помилками для заголовка з 201 символу: `{"title": "x" * 201, "content": "текст"}`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
bad = NoteForm(data={"title": "x" * 201, "content": "текст"})
bad.is_valid()
bad_fields = set(bad.errors)
# END SOLUTION

assert bad_fields == {"title"}, "max_length=200 з моделі перевіряє форма"
for field, errors in bad.errors.items():
    print(field, "→", errors[0])
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Форма `NoteForm(data={'title': 'План'})` — без ключа `content` взагалі. Валідна?

<details>
<summary>Відповідь</summary>

Так. `content` у моделі `blank=True` → поле форми необов'язкове; `cleaned_data['content'] == ''`. А без `title` — помилка «Це поле обов'язкове.».

</details>

---
## 2. Що view передає в шаблон

Тестовий клієнт повертає не лише HTML, а й **контекст шаблону** — словник, який view передав у `render()`.

### Вправа 3

Отримай `/notes/` тестовим клієнтом і збережи заголовки нотаток з контексту (`response.context["notes"]`) у `titles`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.get("/notes/")
titles = [note.title for note in response.context["notes"]]
# END SOLUTION

assert titles == ["Пароль від Wi-Fi", "Вивчити Django Forms", "Купити молоко"], "Meta.ordering = ['-created_at']: нові — першими"
assert response.templates[0].name == "hello_app/note_list.html"
assert "3 нотаток" in response.content.decode(), "{{ notes|length }} у шаблоні"
print("✅ Вправа 3 пройдена")

---
## 3. Створення: помилки і PRG

### Вправа 4

Надішли `POST /notes/new/` з порожнім заголовком: `{"title": "", "content": "текст"}`. Збережи відповідь у `bad_response`.

In [ ]:
before = Note.objects.count()
# YOUR CODE HERE
# BEGIN SOLUTION
bad_response = client.post("/notes/new/", {"title": "", "content": "текст"})
# END SOLUTION

assert bad_response.status_code == 200, "неправильні дані — та сама сторінка з формою"
assert "title" in bad_response.context["form"].errors
assert bad_response.context["form"]["content"].value() == "текст", "введене не губиться"
assert Note.objects.count() == before, "нічого не збережено"
print("✅ Вправа 4 пройдена")

### Вправа 5

Тепер правильні дані: `{"title": "Купити квитки", "content": "Київ — Львів"}`. Відповідь — у `created`. Потім зайди за адресою з `created["Location"]` і збережи повідомлення сторінки (рядками) у `shown`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
created = client.post("/notes/new/", {"title": "Купити квитки", "content": "Київ — Львів"})
page = client.get(created["Location"])
shown = [str(message) for message in page.context["messages"]]
# END SOLUTION

assert created.status_code == 302 and created["Location"] == "/notes/", "PRG: redirect на список"
assert shown == ['Нотатку "Купити квитки" успішно створено!']
assert Note.objects.filter(title="Купити квитки").count() == 1
print("✅ Вправа 5 пройдена")

**🔮 Прогноз:** Що покаже `client.get(created['Location'])` вдруге — те саме повідомлення?

<details>
<summary>Відповідь</summary>

Ні: повідомлення «на одну сторінку». Після першого показу воно зникає, а повторний `GET` нічого не створює й нічого не повідомляє.

</details>

---
## 4. Редагування і видалення

### Вправа 6

Зміни нотатку «Купити квитки» через `POST /notes/<id>/edit/`: заголовок «Купити квитки на потяг», зміст той самий. Відповідь — у `edited`.

In [ ]:
note = Note.objects.get(title="Купити квитки")
# YOUR CODE HERE
# BEGIN SOLUTION
edited = client.post(f"/notes/{note.pk}/edit/", {"title": "Купити квитки на потяг", "content": note.content})
# END SOLUTION

note.refresh_from_db()
assert edited.status_code == 302 and edited["Location"] == f"/notes/{note.pk}/", "після редагування — на деталі"
assert note.title == "Купити квитки на потяг"
assert Note.objects.count() == 4, "редагування не створює нову нотатку"
print("✅ Вправа 6 пройдена")

### Вправа 7

Видали нотатку «Купити молоко»: спершу `GET` сторінки видалення (код — у `confirm_code`, перевір, що нотатка ще є), потім `POST` (відповідь — у `deleted`).

In [ ]:
milk = Note.objects.get(title="Купити молоко")
url = f"/notes/{milk.pk}/delete/"
# YOUR CODE HERE
# BEGIN SOLUTION
confirm_code = client.get(url).status_code
still_there = Note.objects.filter(pk=milk.pk).exists()
deleted = client.post(url)
# END SOLUTION

assert confirm_code == 200 and still_there, "GET лише питає"
assert deleted.status_code == 302 and deleted["Location"] == "/notes/"
assert not Note.objects.filter(pk=milk.pk).exists()
print("✅ Вправа 7 пройдена")

---
## 5. CSRF

Звичайний тестовий клієнт не перевіряє CSRF-токен. `Client(enforce_csrf_checks=True)` поводиться як справжній сервер.

### Вправа 8

Надішли `POST /notes/new/` клієнтом з перевіркою CSRF і без токена. Код відповіді — у `code`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
strict = Client(enforce_csrf_checks=True)
code = strict.post("/notes/new/", {"title": "Без токена"}).status_code
# END SOLUTION

assert code == 403
assert not Note.objects.filter(title="Без токена").exists()
print("✅ Вправа 8 пройдена")

---
## 6. Знайди помилку

```python
def note_create_buggy(request):
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
    return render(request, "hello_app/note_form.html", {"form": form, "title": "Нова нотатка", "action": "Створити"})
```

Після успішного `POST` — сторінка `200`: F5 надішле форму ще раз і створить дубль.

### Вправа 9

Напиши `note_create_fixed(request)` за шаблоном PRG: успіх → `redirect("hello_app:note_list")`, інакше — форма з помилками.

In [ ]:
from django.shortcuts import redirect, render


def note_create_fixed(request):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    form = NoteForm(request.POST or None)
    if request.method == "POST" and form.is_valid():
        form.save()
        return redirect("hello_app:note_list")
    return render(request, "hello_app/note_form.html", {"form": form, "title": "Нова нотатка", "action": "Створити"})
    # END SOLUTION


factory = RequestFactory()
response = note_create_fixed(factory.post("/notes/new/", {"title": "Оплатити інтернет"}))
assert response.status_code == 302 and response["Location"] == "/notes/"
assert Note.objects.filter(title="Оплатити інтернет").exists()
assert note_create_fixed(factory.post("/notes/new/", {"title": ""})).status_code == 200
print("✅ Вправа 9 пройдена")

---
## Самоперевірка

1. Коли форма `method="get"`, а коли `method="post"`?
2. Звідки `NoteForm` знає, що `title` не довший за 200 символів, а `content` необов'язковий?
3. Опиши PRG. Яку проблему він розв'язує?
4. Чому видалення — лише `POST`?
5. Від чого захищає `{% csrf_token %}`?

<details>
<summary>Відповіді</summary>

1. `GET` — пошук і фільтри (нічого не змінює, дані в URL); `POST` — створення, зміна, видалення.
2. З моделі: `ModelForm` будує поля з `CharField(max_length=200)` і `TextField(blank=True)`.
3. Post / Redirect / Get: після успішного `POST` — `302`, браузер робить `GET`. F5 повторить лише `GET`, дублів немає.
4. `GET` не має змінювати дані: посилання відкривають роботи, попереднє завантаження, розширення.
5. Від CSRF: чужий сайт не знає секретного токена сторінки, тож його підроблений `POST` отримає `403`.

</details>

## Далі

- Книга курсу: [Урок 34](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/) — рефакторинг 1 (Bootstrap CRUD) і рефакторинг 2 (crispy dashboard), діфи й скріншоти.
- Запусти проєкт: `python manage.py runserver` у папці `django_bootstrap_project` → `http://127.0.0.1:8000/notes/`.
- Спробуй рефакторинг 2 сам: переведи `note_form.html` цього проєкту на `{% crispy form %}` — завдання «Спробуй самостійно» на сторінці уроку.
- **Урок 35** — наступний рефакторинг: REST API до нотаток (DRF).